# Math 11 · Attention and Transformer Mathematics

**Companion lessons:** Lessons 37-45, Papers 10-12

> **Math goal:** understand the objects, assumptions, derivation, and failure modes behind the algorithms, not just memorize formulas.

## Scaled dot-product attention

Given

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V,
$$

attention is

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

For sequence length $T$, $QK^T$ is $T\times T$: every query position scores every key position.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import torch, math
B,H,T,D=2,4,6,8
Q=torch.randn(B,H,T,D); K=torch.randn(B,H,T,D); V=torch.randn(B,H,T,D)
scores=Q@K.transpose(-2,-1)/math.sqrt(D)
weights=torch.softmax(scores,-1)
out=weights@V
print("scores",scores.shape,"weights",weights.shape,"out",out.shape)

## Why divide by $\sqrt{d_k}$?

If components of $q$ and $k$ are independent with mean 0 and variance 1,

$$
q^Tk=\sum_{i=1}^{d_k}q_ik_i
$$

has variance approximately $d_k$.

Dividing by $\sqrt{d_k}$ keeps score variance roughly constant as dimension grows, preventing softmax from becoming excessively saturated.

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
for d in [4,16,64,256]:
    q=rng.normal(size=(20000,d)); k=rng.normal(size=(20000,d))
    dots=np.sum(q*k,axis=1)
    print(d,"raw std",dots.std(),"scaled std",(dots/np.sqrt(d)).std())

## Multi-head attention

If model dimension is $d_{model}$ and there are $h$ heads, a common choice is

$$
d_k=d_v=d_{model}/h.
$$

Different heads can learn different subspace projections.

## Positional information

Self-attention without positional encoding is permutation-equivariant: permuting tokens permutes outputs in the same way.

Position encodings break this symmetry so sequence order matters.

In [ ]:
# Permutation-equivariance demonstration using attention without positions
torch.manual_seed(0)
X=torch.randn(1,5,8)
Q=K=V=X
def attn(x):
    s=x@x.transpose(-2,-1)/math.sqrt(x.shape[-1])
    return torch.softmax(s,-1)@x
perm=torch.tensor([2,0,4,1,3])
a=attn(X)
b=attn(X[:,perm])
print(torch.allclose(b,a[:,perm],atol=1e-5))

### Activity
Compute attention entropy with and without scaling across `d_k`. Then add a causal mask and show which entries become inaccessible.

## Derivation checkpoint
In a new Markdown cell, re-derive the main result without copying the notebook. State every variable's shape and every assumption used.

## Engineering checkpoint
Explain which approximation or assumption is most likely to break in a real system, and how you would detect that failure from data.